# From Data to Decision: Diagnosing & Fixing SaaS Customer Churn

**A product analytics project combining predictive modeling, NLP, and product prioritization**

Dataset: real SaaS customer churn data (2,500 records) with account age, login frequency,
daily usage, and support ticket text.

**Structure of this notebook:**
1. Setup
2. EDA & Sentiment Analysis (Diagnose)
3. Predictive Model & SHAP Interpretability (Diagnose)
4. Model Validation — ROC/PR curves, Cross-Validation, Risk-Scored Customer List
5. Product Prioritization & A/B Test Design (Decide)
6. Business Impact & Conclusion (Prove)


## 1. Load Data & Extract Sentiment

We use VADER sentiment analysis on each customer's most recent support ticket text,
combined with structured usage/login data, to understand what actually drives churn.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
!pip install vaderSentiment shap xgboost statsmodels -q
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

sns.set_style("whitegrid")
analyzer = SentimentIntensityAnalyzer()

train = pd.read_csv("/content/drive/MyDrive/train.csv")
test = pd.read_csv("/content/drive/MyDrive/test_.csv")

def add_sentiment(df):
    scores = df["Last_Support_Ticket"].apply(lambda t: analyzer.polarity_scores(str(t)))
    df["sentiment_compound"] = scores.apply(lambda s: s["compound"])
    df["sentiment_neg"] = scores.apply(lambda s: s["neg"])
    df["sentiment_pos"] = scores.apply(lambda s: s["pos"])
    return df

train = add_sentiment(train)
test = add_sentiment(test)

train.to_csv("/content/drive/MyDrive/real_train_with_sentiment.csv", index=False)
test.to_csv("/content/drive/MyDrive/real_test_with_sentiment.csv", index=False)

print("Train shape:", train.shape, "| Test shape:", test.shape)
print("\nOverall churn rate:", round(train["Churn"].mean(), 3))
print("\nChurn rate by Login Frequency:")
print(train.groupby("Login_Frequency")["Churn"].mean().round(3))

train["usage_bucket"] = pd.cut(train["Daily_Usage_Mins"], bins=[-1,15,40,1000],
                                 labels=["Low(<15m)","Medium(15-40m)","High(40m+)"])
print("\nChurn rate by usage bucket:")
print(train.groupby("usage_bucket")["Churn"].agg(["mean","count"]))

train["sentiment_bucket"] = pd.cut(train["sentiment_compound"], bins=[-1,-0.3,0.3,1],
                                     labels=["Negative","Neutral","Positive"])
print("\nChurn rate by sentiment bucket:")
print(train.groupby("sentiment_bucket")["Churn"].agg(["mean","count"]))

Train shape: (2000, 11) | Test shape: (500, 11)

Overall churn rate: 0.368

Churn rate by Login Frequency:
Login_Frequency
Daily     0.130
Rarely    0.855
Weekly    0.423
Name: Churn, dtype: float64

Churn rate by usage bucket:
                    mean  count
usage_bucket                   
Low(<15m)       0.814925    670
Medium(15-40m)  0.181996    511
High(40m+)      0.119658    819

Churn rate by sentiment bucket:
                      mean  count
sentiment_bucket                 
Negative          0.834821    224
Neutral           0.342969   1280
Positive          0.223790    496


/tmp/ipykernel_1501/1608527438.py:37: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(train.groupby("usage_bucket")["Churn"].agg(["mean","count"]))
/tmp/ipykernel_1501/1608527438.py:42: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(train.groupby("sentiment_bucket")["Churn"].agg(["mean","count"]))


**Key finding #1:** A third of customers (33.5%) use the product less than 15 minutes/day
— and **81.5% of them churn.** This is the single biggest lever in the dataset.

**Key finding #2:** Only 11% of customers file a negative-sentiment support ticket, but
when they do, churn hits **83.5%** — a smaller but near-certain warning sign that a
structured-data-only model would completely miss.


In [5]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

order = ["Daily", "Weekly", "Rarely"]
churn_by_login = train.groupby("Login_Frequency")["Churn"].mean().reindex(order)
axes[0].bar(order, churn_by_login.values, color=["#2e7d32", "#f9a825", "#c62828"])
axes[0].set_title("Churn Rate by Login Frequency")
axes[0].set_ylabel("Churn Rate")
for i, v in enumerate(churn_by_login.values):
    axes[0].text(i, v + 0.02, f"{v:.0%}", ha="center", fontweight="bold")

sns.kdeplot(data=train, x="sentiment_compound", hue="Churn", fill=True, ax=axes[1],
            palette={0: "#2e7d32", 1: "#c62828"}, common_norm=False)
axes[1].set_title("Support Ticket Sentiment: Churned vs Retained")
axes[1].set_xlabel("Sentiment (compound, -1=negative, +1=positive)")

sample = train.sample(min(600, len(train)), random_state=1)
colors = sample["Churn"].map({0: "#2e7d32", 1: "#c62828"})
axes[2].scatter(sample["Account_Age_Days"], sample["Daily_Usage_Mins"], c=colors, alpha=0.5, s=18)
axes[2].set_title("Usage vs Account Age (red = churned)")
axes[2].set_xlabel("Account Age (days)")
axes[2].set_ylabel("Daily Usage (mins)")

plt.tight_layout()
plt.savefig("/content/drive/MyDrive/real_eda_overview.png", dpi=150)
plt.show()

## 2. Predictive Model & SHAP Interpretability

We train an XGBoost classifier combining usage metrics, account age, and sentiment
features to predict churn, then use SHAP to explain *why* the model makes each
prediction — turning a black-box model into an interpretable, business-usable tool.


In [11]:
import xgboost as xgb
import shap
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
train["Login_Frequency_enc"] = le.fit_transform(train["Login_Frequency"])
test["Login_Frequency_enc"] = le.transform(test["Login_Frequency"])

model_features = ["Account_Age_Days", "Daily_Usage_Mins", "Login_Frequency_enc",
                   "sentiment_compound", "sentiment_neg", "sentiment_pos"]

X_train, y_train = train[model_features], train["Churn"]
X_test, y_test = test[model_features], test["Churn"]

model = xgb.XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.08,
    subsample=0.9, colsample_bytree=0.9, eval_metric="logloss",
    random_state=42
)
model.fit(X_train, y_train)

pred = model.predict(X_test)
proba = model.predict_proba(X_test)[:, 1]

# ---- Risk-scored customer export ----
test_out = test.copy()
test_out["churn_risk_score"] = proba

risk_list = test_out.sort_values("churn_risk_score", ascending=False)[
    ["Customer_ID", "Name", "Login_Frequency", "Daily_Usage_Mins",
     "Last_Support_Ticket", "churn_risk_score"]
]

risk_list.to_csv("/content/drive/MyDrive/top_at_risk_customers.csv", index=False)
print("Top 10 highest-risk customers:")
print(risk_list.head(10).to_string(index=False))

print("=== Classification Report ===")
print(classification_report(y_test, pred, digits=3))
print("ROC-AUC:", round(roc_auc_score(y_test, proba), 3))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, pred))

Top 10 highest-risk customers:
                         Customer_ID             Name Login_Frequency  Daily_Usage_Mins                                          Last_Support_Ticket  churn_risk_score
c07c0a8a-bac0-47f9-b93d-57d61490a610     Dorothy Rose          Rarely                 1    I've been waiting for support for 3 days. I'm cancelling.          0.997601
17d10c92-8ca7-45d3-9645-6164b087b857   Timothy Porter          Rarely                 1    I've been waiting for support for 3 days. I'm cancelling.          0.995693
dbf0c0e2-c5d1-411a-bbb2-f7944332f64d      Ryan Guerra          Rarely                 1     The UI is too confusing. I can't find the export button.          0.988029
d4b5096b-14ba-4028-8d30-162c71f7dffb  Joshua Thompson          Rarely                 1    I've been waiting for support for 3 days. I'm cancelling.          0.987177
e21cd418-0533-4a73-8b11-60d01aa12377     Tonya Castro          Weekly                 1            The app crashes every time I try to

In [7]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

X_test_display = X_test.copy()
X_test_display.columns = ["Account Age (days)", "Daily Usage (mins)", "Login Frequency (enc)",
                           "Sentiment (compound)", "Sentiment (negative)", "Sentiment (positive)"]

plt.figure()
shap.summary_plot(shap_values, X_test_display, show=False, plot_size=(9, 5))
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()

mean_abs_shap = pd.Series(np.abs(shap_values).mean(axis=0), index=X_test_display.columns)
mean_abs_shap = mean_abs_shap.sort_values(ascending=False)
print("Mean |SHAP value| per feature (driver strength):")
print(mean_abs_shap.round(4))

Mean |SHAP value| per feature (driver strength):
Daily Usage (mins)       1.4082
Account Age (days)       0.3466
Sentiment (compound)     0.2890
Sentiment (negative)     0.1577
Login Frequency (enc)    0.0984
Sentiment (positive)     0.0669
dtype: float32


**Driver ranking:** usage depth is, by a wide margin, the strongest churn signal.
Sentiment adds meaningful, complementary signal as a high-precision early-warning
flag for a smaller at-risk segment, but doesn't move the needle as much as
engagement depth on its own.


## 3. Model Validation

Before trusting this model's output, we check it two ways:
1. **Cross-validation** — is the ROC-AUC stable across different data splits, or did we get lucky?
2. **ROC / Precision-Recall curves** — visual proof of model quality across thresholds.

We also export a **risk-scored customer list** — the actual usable output a retention
team would act on, not just a summary metric.


In [8]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay

cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring="roc_auc")
print(f"5-fold CV ROC-AUC: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
RocCurveDisplay.from_estimator(model, X_test, y_test, ax=axes[0])
axes[0].set_title("ROC Curve")
PrecisionRecallDisplay.from_estimator(model, X_test, y_test, ax=axes[1])
axes[1].set_title("Precision-Recall Curve")
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/model_curves.png", dpi=150)
plt.show()

5-fold CV ROC-AUC: 0.848 ± 0.024


In [ ]:
test_out = test.copy()
test_out["churn_risk_score"] = proba
risk_list = test_out.sort_values("churn_risk_score", ascending=False)[
    ["Customer_ID", "Name", "Login_Frequency", "Daily_Usage_Mins",
     "Last_Support_Ticket", "churn_risk_score"]
]
risk_list.to_csv("/content/drive/MyDrive/top_at_risk_customers.csv", index=False)
print("Top 10 highest-risk customers:")
risk_list.head(10)

Top 10 highest-risk customers:
                          Customer_ID             Name Login_Frequency  Daily_Usage_Mins                                          Last_Support_Ticket  churn_risk_score
c07c0a8a-bac0-47f9-b93d-57d61490a610     Dorothy Rose          Rarely                 1   I've been waiting for support for 3 days. I'm cancelling.          0.997601
17d10c92-8ca7-45d3-9645-6164b087b857   Timothy Porter          Rarely                 1   I've been waiting for support for 3 days. I'm cancelling.          0.995693
dbf0c0e2-c5d1-411a-bbb2-f7944332f64d      Ryan Guerra          Rarely                 1    The UI is too confusing. I can't find the export button.          0.988029
d4b5096b-14ba-4028-8d30-162c71f7dffb  Joshua Thompson          Rarely                 1   I've been waiting for support for 3 days. I'm cancelling.          0.987177
e21cd418-0533-4a73-8b11-60d01aa12377     Tonya Castro          Weekly                 1           The app crashes every time I try to upl

## 4. Product Prioritization & A/B Test Design

Each SHAP driver is translated into a candidate product intervention, scored with a
RICE framework (Reach x Impact x Confidence / Effort). We then size an A/B test for
the top-ranked intervention using a real statistical power calculation.


In [9]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

interventions = pd.DataFrame([
    {"name": "Usage-triggered nurture program", "reach": 8, "impact": 9, "confidence": 8, "effort": 4},
    {"name": "Automated re-engagement email",   "reach": 9, "impact": 5, "confidence": 6, "effort": 2},
    {"name": "Early-account onboarding push",   "reach": 6, "impact": 7, "confidence": 7, "effort": 3},
    {"name": "Sentiment-triggered escalation",  "reach": 4, "impact": 8, "confidence": 8, "effort": 3},
])
interventions["rice_score"] = (
    interventions.reach * interventions.impact * interventions.confidence
) / interventions.effort
print("=== RICE Prioritization ===")
print(interventions.sort_values("rice_score", ascending=False).to_string(index=False))

# A/B test sample size for the top intervention
baseline_churn = 0.815   # low-usage segment baseline
target_churn = 0.665     # 15pt reduction goal
effect_size = proportion_effectsize(baseline_churn, target_churn)

analysis = NormalIndPower()
n_per_group = analysis.solve_power(effect_size, alpha=0.05, power=0.8, ratio=1)
print(f"\n=== A/B Test Sample Size ===")
print(f"Baseline churn (low-usage segment): {baseline_churn:.1%}")
print(f"Target churn (15pt reduction):      {target_churn:.1%}")
print(f"Required sample size per arm: {n_per_group:.0f}")

=== RICE Prioritization ===
                           name  reach  impact  confidence  effort  rice_score
Usage-triggered nurture program      8       9           8       4  144.000000
  Automated re-engagement email      9       5           6       2  135.000000
  Early-account onboarding push      6       7           7       3   98.000000
 Sentiment-triggered escalation      4       8           8       3   85.333333

=== A/B Test Sample Size ===
Baseline churn (low-usage segment): 81.5%
Target churn (15pt reduction):      66.5%
Required sample size per arm: 132


## 5. Business Impact & Conclusion

In [10]:
total_customers = 10000
segment_pct = 0.335
arpu_monthly = 50

def annual_impact(churn_reduction_pts):
    affected = total_customers * segment_pct
    retained_customers = affected * churn_reduction_pts
    return retained_customers * arpu_monthly * 12

print(f"Projected Annual Revenue Impact")
print(f"(Assumes {total_customers:,} customers, {segment_pct:.1%} in low-usage segment, ${arpu_monthly}/mo ARPU)\n")
for label, pts in [("Downside", 0.05), ("Base case", 0.15), ("Upside", 0.25)]:
    print(f"{label} ({pts:.0%} churn reduction): ${annual_impact(pts):,.0f}")

Projected Annual Revenue Impact
(Assumes 10,000 customers, 33.5% in low-usage segment, $50/mo ARPU)

Downside (5% churn reduction): $100,500
Base case (15% churn reduction): $301,500
Upside (25% churn reduction): $502,500


### Conclusion

- **Usage depth is the dominant churn driver**: customers averaging under 15 minutes/day
  churn at 81.5%, vs. 12% for daily active users.
- **Support-ticket sentiment is a smaller but highly precise secondary signal**: only 11%
  of customers file a negative-sentiment ticket, but 83.5% of them churn.
- **Model**: XGBoost + SHAP achieves 0.806 ROC-AUC (0.848 ± 0.024 under 5-fold CV) —
  stable, not a lucky split.
- **Recommendation**: Pilot a usage-triggered nurture program (highest RICE score,
  144) targeting the low-usage segment, validated via a properly powered A/B test
  (~132 customers/arm). Projected to protect **$301K in annual revenue** (base case)
  against a ~4-person-week engineering cost.
- **Deliverable**: a risk-scored customer list (`top_at_risk_customers.csv`) that a
  retention/success team could act on starting today.



